> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


## 实验四：GEMM基础版算子开发与验证


建议学时：4学时


## 实验任务


1.任务描述


本实验实现一个采用FP32数据类型的GEMM基础版算子。输入矩阵为A[M,K]和B[K,N]，输出矩阵为C[M,N]。基础版采用直观的三层循环完成矩阵乘法，用于验证矩阵索引、tiling传参和PyTorch注册链路。


2.学习目标


完成本实验后，学生应能够：


• 理解GEMM在深度学习线性层中的作用。


• 掌握行主序矩阵的地址计算和K维累加逻辑。


• 能够实现基础版GEMM Kernel并注册为PyTorch接口。


• 能够将自定义GEMM用于Qwen2.5线性层替换验证。


## 任务准备


1.算子定义


GEMM基础版算子实现通用矩阵乘法，输入为A[M,K]和B[K,N]，输出为C[M,N]。该算子的核心公式如下。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">说明</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">算子名称</td>
<td style="text-align:left;">GEMM基础版算子</td>
</tr>
<tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">A[M,K]和B[K,N]，均为float32类型NPU连续张量</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">C[M,N]</td>
</tr>
<tr>
<td style="text-align:left;">计算语义</td>
<td style="text-align:left;">对输出矩阵每个元素沿K维累加乘积</td>
</tr>
<tr>
<td style="text-align:left;">模型位置</td>
<td style="text-align:left;">用于替换Qwen2.5线性层中的矩阵乘法部分</td>
</tr>
</tbody></table>


2.GEMM简介


GEMM是通用矩阵乘法，也是大模型线性层的核心计算。对于线性层来说，主要计算可以写成如下形式。


将三维输入展平成二维矩阵后，就可以用GEMM完成线性层中的乘法部分。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">A[M,K] × B[K,N] = C[M,N] M：批量与序列展平后的行数 K：输入特征维度 N：输出特征维度</th>
</tr>
</thead>
</table>


基础版GEMM使用最容易理解的写法：遍历输出矩阵的每个位置，再沿K维累加。


3.算子与接口定义


本实验把基础版GEMM封装为PyTorch自定义算子。接口侧负责检查输入形状、生成tiling、启动kernel并返回输出张量。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">接口项</th>
<th style="text-align:left;">约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">PyTorch调用接口</td>
<td style="text-align:left;">torch.ops.gemm_custom.gemm(a, b)</td>
</tr>
<tr>
<td style="text-align:left;">device侧入口</td>
<td style="text-align:left;">gemm_baseline_kernel</td>
</tr>
<tr>
<td style="text-align:left;">核心实现文件</td>
<td style="text-align:left;">op_kernel/gemm_baseline_kernel.h</td>
</tr>
<tr>
<td style="text-align:left;">任务划分文件</td>
<td style="text-align:left;">op_kernel/gemm_tiling.h</td>
</tr>
<tr>
<td style="text-align:left;">注册文件</td>
<td style="text-align:left;">torch_extension/gemm_torch_register.asc</td>
</tr>
</tbody></table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">A[M,K]和B[K,N]均为float32类型NPU连续张量</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">C[M,N]</td>
</tr>
<tr>
<td style="text-align:left;">计算公式</td>
<td style="text-align:left;">见前置知识中的公式说明</td>
</tr>
<tr>
<td style="text-align:left;">调用方式</td>
<td style="text-align:left;">torch.ops.gemm_custom.gemm(a, b)</td>
</tr>
<tr>
<td style="text-align:left;">实现特点</td>
<td style="text-align:left;">按行划分任务，使用朴素K维累加</td>
</tr>
</tbody></table>


4.实验环境准备


本实验在Ascend NPU云服务器上完成，使用CANN工具链、AscendC和PyTorch NPU环境。基础版与优化版建议放在不同工程目录中，构建和测试也尽量在新的Python进程中执行，避免torch.library重复注册。


进入工程目录后，先加载CANN环境变量，再检查环境、构建工程并设置动态库搜索路径。CANN安装路径以云服务器实际配置为准。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd src/GemmBaselineExperiment source $QWEN_OPS_CODE_ROOT/scripts/setup_cannlab_env.sh bash scripts/check_env.sh bash scripts/build.sh export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH</th>
</tr>
</thead>
</table>


## 任务实施


## 步骤一：准备工程结构


本步骤介绍工程在文件层面的组织方式。基础版工程将算子内核、PyTorch注册、独立直调、脚本和测试分开存放，使host侧、device侧和模型侧职责清晰。复制工程后统一修改命名，是为了让构建产物、Python接口和运行脚本始终对应同一套算子实现。


基础版工程放在src/GemmBaselineExperiment。GEMM实验的核心文件包括算子内核实现、tiling结构、PyTorch注册代码、独立直调程序和测试脚本。工程中的gemm_custom命名空间、算子内核入口名、动态库名和torch_extension/__init__.py加载路径共同构成PyTorch侧的调用入口。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">GemmBaselineExperiment/ ├── op_kernel/ │ ├── gemm_baseline_kernel.cpp │ ├── gemm_baseline_kernel.h │ └── gemm_tiling.h ├── torch_extension/ │ ├── <strong>init</strong>.py │ └── gemm_torch_register.asc ├── verify_kernel_launch/ ├── scripts/ └── tests/</th>
</tr>
</thead>
</table>


## 步骤二：定义矩阵规模和任务划分


本步骤说明GEMM基础版如何把矩阵规模映射到device任务。m、n和k分别对应输出行数、输出列数和累加维度，rowsPer算核决定每个算核处理多少行。这个步骤的作用是让矩阵乘法的工作范围能够直接在kernel中展开。


基础版tiling保存m、n、k以及按行划分后的算核。矩阵均按行主序存储，地址关系为A[row*k+kk]、B[kk*n+col]、C[row*n+col]。这三个地址公式把二维矩阵位置转换成一维内存地址，是GEMM代码展开的基础。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct GemmBaselineTiling { uint32_t m = 0; uint32_t n = 0; uint32_t k = 0; uint32_t coreNum = 1; uint32_t rowsPerCore = 0; uint32_t reserved0 = 0; uint32_t reserved1 = 0; uint32_t reserved2 = 0; }; #pragma pack(pop)</th>
</tr>
</thead>
</table>


## 步骤三：实现朴素矩阵乘法


本步骤展示GEMM基础版的三重循环结构。kernel按行、列、K维依次展开，每次对一个输出元素做完整的累加，最终写回到C[row,col]。它的功能是把矩阵乘法的数学定义直接翻译成代码。


基础版每个算核处理一段连续的行。对每个输出元素C[row,col]，算子内核沿K维逐项累加。代码结构接近CPU上的朴素矩阵乘法，便于理解，但全局内存访问频繁，性能不会接近原生矩阵乘实现。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;"><strong>aicore</strong> inline void Process() { const uint32_t coreId = GetBlockIdx(); if (coreId &gt;= coreNum_) { return; } const uint32_t rowBegin = coreId * rowsPerCore_; uint32_t rowEnd = rowBegin + rowsPerCore_; if (rowEnd &gt; m_) { rowEnd = m_; } for (uint32_t row = rowBegin; row &lt; rowEnd; ++row) { for (uint32_t col = 0; col &lt; n_; ++col) { float acc = 0.0f; for (uint32_t kk = 0; kk &lt; k_; ++kk) { const float av = aGm_.GetValue(row * k_ + kk); const float bv = bGm_.GetValue(kk * n_ + col); acc += av * bv; } cGm_.SetValue(row * n_ + col, acc); } } }</th>
</tr>
</thead>
</table>


## 步骤四：注册PyTorch接口


注册侧约束A[M,K]和B[K,N]的形状关系，输出C[M,N]。这里不做矩阵转置，kernel读取的是原始B[K,N]布局，因此注册侧传入的B矩阵和device侧的地址计算保持一致。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">GemmBaselineTiling BuildTiling(uint32_t m, uint32_t n, uint32_t k) { const uint32_t coreNum = std::max(1U, std::min(kDefaultBlockDim, m)); GemmBaselineTiling tiling {}; tiling.m = m; tiling.n = n; tiling.k = k; tiling.coreNum = coreNum; tiling.rowsPerCore = (m + coreNum - 1U) / coreNum; return tiling; } torch::Tensor gemm_baseline_npu(const torch::Tensor &amp;a, const torch::Tensor &amp;b) { CheckInput(a, b); const int32_t deviceIndex = a.device().index() &lt; 0 ? 0 : a.device().index(); const c10::DeviceGuard deviceGuard(a.device()); auto c = torch::empty({a.size(0), b.size(1)}, a.options()); const GemmBaselineTiling tiling = BuildTiling( static_cast&lt;uint32_t&gt;(a.size(0)), static_cast&lt;uint32_t&gt;(b.size(1)), static_cast&lt;uint32_t&gt;(a.size(1))); aclrtStream stream = c10_npu::getCurrentNPUStream(deviceIndex).stream(true); const uint32_t launchRet = ACLRT_LAUNCH_KERNEL(gemm_baseline_kernel)( tiling.coreNum, stream, static_cast&lt;uint8_t *&gt;(a.data_ptr()), static_cast&lt;uint8_t *&gt;(b.data_ptr()), static_cast&lt;uint8_t *&gt;(c.data_ptr()), static_cast&lt;uint8_t *&gt;(workspaceD), static_cast&lt;uint8_t *&gt;(tilingD)); TORCH_CHECK(launchRet == 0, &quot;gemm_baseline_kernel launch failed&quot;); CHECK_ACL_THROW(aclrtSynchronizeStream(stream)); return c; }</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_torch_op.py --m 128 --k 1024 --n 512 --atol 1e-3 --rtol 1e-3 ./out/bin/gemm_baseline_standalone \ --m 128 \ --k 1024 \ --n 512 \ --block-dim 8 \ --warmup 10 \ --repeat 20 \ --rounds 5</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_qwen_linear.py --batch 1 --seq 128 --hidden 1024 --out 512 python3 tests/compare_qwen_native.py \ --model $QWEN_OPS_MODEL_PATH \ --repeat 3 \ --attn-implementation eager</th>
</tr>
</thead>
</table>


## Torch NPU 注册层兼容性（与工程源码同步）

当前 `torch_extension/gemm_torch_register.asc` 先检查 `a` 与 `b` 位于同一 NPU，再以输入设备建立守卫并取得 PyTorch 当前流：

```cpp
TORCH_CHECK(a.device() == b.device(), "a and b must be on the same NPU device");
const int32_t deviceIndex = a.device().index() < 0 ? 0 : a.device().index();
const c10::DeviceGuard deviceGuard(a.device());
aclrtStream stream = c10_npu::getCurrentNPUStream(deviceIndex).stream(true);
```

这样算子会加入调用方当前 NPU stream，不创建私有 ACL stream。CMake 还会优先使用 torch_npu 随包提供、与当前版本匹配的 ACL 头文件：

```cmake
${TORCH_NPU_INCLUDE}/third_party/acl/inc
${ASCEND_CANN_PACKAGE_PATH}/include
```

该顺序用于避免较新 CANN 系统头与当前 torch_npu 扩展头发生类型不匹配。


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

if [[ -z "${QWEN_OPS_CODE_ROOT:-}" ]]; then
    QWEN_OPS_REPO_ROOT="$(git rev-parse --show-toplevel 2>/dev/null)" || { echo "无法定位仓库根目录，请设置 QWEN_OPS_CODE_ROOT" >&2; exit 1; }
    QWEN_OPS_CODE_ROOT="$QWEN_OPS_REPO_ROOT/contrib/tutorials/qwen_ops"
fi
source "$QWEN_OPS_CODE_ROOT/scripts/setup_cannlab_env.sh"
cd "$QWEN_OPS_CODE_ROOT/04_gemm_baseline/src/GemmBaselineExperiment"
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

if [[ -z "${QWEN_OPS_CODE_ROOT:-}" ]]; then
    QWEN_OPS_REPO_ROOT="$(git rev-parse --show-toplevel 2>/dev/null)" || { echo "无法定位仓库根目录，请设置 QWEN_OPS_CODE_ROOT" >&2; exit 1; }
    QWEN_OPS_CODE_ROOT="$QWEN_OPS_REPO_ROOT/contrib/tutorials/qwen_ops"
fi
source "$QWEN_OPS_CODE_ROOT/scripts/setup_cannlab_env.sh"
cd "$QWEN_OPS_CODE_ROOT/04_gemm_baseline/src/GemmBaselineExperiment"
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/gemm_baseline_standalone --m 128 --k 1024 --n 512 --block-dim 8 --warmup 2 --repeat 5 --rounds 3


## 任务拓展


完成基础实验后，可以继续围绕以下方向拓展：


• 在相同输入规模下对比基础版、优化版和原生算子的耗时。


• 调整任务划分和分块参数，观察正确性、吞吐和尾块处理是否变化。


• 将单算子测试、独立直调测试和模型替换测试的结果放在一起分析，区分算子内核内部耗时与端到端调度开销。


• 进一步尝试双缓冲、异步搬运、多级流水、FP16/BF16支持或更贴近硬件矩阵单元的实现。


## 实验总结


通过本实验，可以掌握GEMM基础版算子的完整开发链路。基础版重点在于正确理解矩阵形状、行主序地址计算和K维累加，为后续分块、连续布局和向量化归约优化打基础。


基础版GEMM把矩阵乘法的数学定义原封不动地写成代码，因此最适合讲清楚行主序、K维累加和线性层替换。它虽然慢，但提供了一个足够直接的基线，让后续优化的收益能够被精确比较出来。
